In [1]:
%cd ../..

/Users/bezha/PycharmProjects/TripleStreams


In [2]:
import torch, torch.nn as nn
from model import load_model, FlexControlTripleStreamsVAE

model = load_model(
    model_path="eval/Post-Training/ControlConfig2/CoM_RelDen_0_5_step_1075692.pth",
    model_class=FlexControlTripleStreamsVAE,
    is_evaluating=True
).eval()

✅ Using config from model file
[None, None] [None, None, None, None]
🎉 Successfully loaded FlexControlTripleStreamsVAE


In [3]:
import torch, torch.nn as nn, torch.onnx

class EncoderAllWrapper(nn.Module):
    def __init__(self, vae):
        super().__init__()
        self.vae = vae
    def forward(self, flat_hvo_groove, com_magnitude, com_angle):
        mu, log_var, latent_z, memory = self.vae.encode_all(
            flat_hvo_groove, encoding_control_tokens=torch.cat([com_magnitude, com_angle], dim=1),
        )
        return latent_z

wrapper = EncoderAllWrapper(model)

flat_hvo_groove = torch.randn(1, 32, 3, dtype=torch.float32)
com_magnitude = torch.randn(1, 1, dtype=torch.float32)
com_angle = torch.randn(1, 1, dtype=torch.float32)

torch.onnx.export(
    wrapper,
    (flat_hvo_groove, com_magnitude, com_angle),
    "encoder.onnx",
    input_names=["flat_hvo_groove", "com_magnitude", "com_angle"],
    output_names=["latent_z"],
    dynamic_axes={
        "flat_hvo_groove": {0: "N"},
        "encoding_control1_token": {0: "N"},
        "encoding_control2_token": {0: "N"},
        "latent_z": {0: "N"},
    },
    opset_version=17,
    do_constant_folding=True,
)


/var/folders/lr/8ctpqx7n6m54ydpt525nf6q80000gn/T/ipykernel_83982/1811274929.py:19: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(
/Users/bezha/anaconda3/envs/TripleStreams/lib/python3.9/site-packages/torch/onnx/utils.py:1853: UserWarning: Provided key encoding_control1_token for dynamic axes is not a valid input/output name
  warnings.warn(
/Users/bezha/anaconda3/envs/TripleStreams/lib/python3.9/site-packages/torch/onnx/utils.py:1853: UserWarn

In [4]:
import torch
import torch.nn as nn

class DecoderWrapper(nn.Module):
    def __init__(self, vae):
        super().__init__()
        self.vae = vae

    def forward(self, latent_z, N_active_steps, s1_rel_den, s2_rel_den, s3_rel_den,
                voice_thresholds, voice_max_count_allowed):
        # Concatenate control tokens
        decoding_control_tokens = torch.cat([N_active_steps, s1_rel_den, s2_rel_den, s3_rel_den], dim=1)

        h, v, o = self.vae.sample(
            latent_z=latent_z,
            decoding_control_tokens=decoding_control_tokens,
            voice_thresholds=voice_thresholds,
            voice_max_count_allowed=voice_max_count_allowed,
            sampling_mode=0
        )
        return h, v, o

def export_decoder_to_onnx(model, output_path="decoder.onnx"):
    """
    Export the decoder model to ONNX format.

    Args:
        model: The VAE model to wrap and export
        output_path: Path where to save the ONNX model

    Returns:
        bool: True if export successful, False otherwise
    """
    # Create wrapper
    wrapper = DecoderWrapper(model)

    # Use non-zero test inputs (this was the key to making it work)
    test_inputs = (
        torch.randn(1, 256, dtype=torch.float32) * 0.1,     # Small random latent
        torch.tensor([[16.0]], dtype=torch.float32),         # N_active_steps = 16
        torch.tensor([[0.8]], dtype=torch.float32),          # s1_rel_den = 0.8
        torch.tensor([[0.6]], dtype=torch.float32),          # s2_rel_den = 0.6
        torch.tensor([[0.4]], dtype=torch.float32),          # s3_rel_den = 0.4
        torch.tensor([0.5, 0.5, 0.5], dtype=torch.float32), # voice_thresholds
        torch.tensor([32, 32, 32], dtype=torch.int64),       # voice_max_count_allowed
    )

    try:
        torch.onnx.export(
            wrapper,
            test_inputs,
            output_path,
            input_names=[
                "latent_z", "N_active_steps", "s1_rel_den", "s2_rel_den", "s3_rel_den",
                "voice_thresholds", "voice_max_count_allowed"
            ],
            output_names=["h", "v", "o"],
            dynamic_axes={
                "latent_z": {0: "batch"},
                "N_active_steps": {0: "batch"},
                "s1_rel_den": {0: "batch"},
                "s2_rel_den": {0: "batch"},
                "s3_rel_den": {0: "batch"},
            },
            opset_version=17,
            do_constant_folding=False,
            verbose=False
        )
        return True
    except Exception:
        return False

# Example usage:
success = export_decoder_to_onnx(model, "decoder.onnx")
if success:
    print("Export successful!")
else:
    print("Export failed!")

/var/folders/lr/8ctpqx7n6m54ydpt525nf6q80000gn/T/ipykernel_83982/928707277.py:49: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(
/Users/bezha/PycharmProjects/TripleStreams/model/FlexControlTripleStreamsVAE/components.py:625: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the t

Export successful!


In [5]:
import onnx, onnxruntime as ort, numpy as np
onnx.checker.check_model(onnx.load("encoder.onnx"))
ort.InferenceSession("encoder.onnx")  # loads fine → you’re good
onnx.checker.check_model(onnx.load("decoder.onnx"))
ort.InferenceSession("decoder.onnx")  # loads fine → you’re good